# Lab 2 Agent Pipeline (experiments and retrieval over your saved pipeline)
**Objective:** direct an agent through classification experiments and retrieval comparisons over the pipeline you built and saved in `DM2026-Lab2-Master.ipynb`, and audit what it reports.

This notebook comes after the Master. There you built five representations of the same tweets, trained classifiers on them, and measured retrieval; the Master's Section 9 saved all of it to `lab2_workspace/`. Here an agent runs experiments on that saved pipeline under your direction. It is the same kind of agent as in the Lab 1 agentic notebooks, built on the same `agent_pipeline/` package: one tool call per message, a mandatory plain-text comment after every tool call, an approval step before any tool that computes something new, and a check on every number it states.

Nothing here re-teaches the Master. The agent runs and compares; your part is to decide what is worth running, one controlled step at a time, and to check every claim against the tool output that produced it.

## Table of Contents
0. Setup: your name, a backend, and its API key
1. Load the saved pipeline
2. What the agent can do, and what to check
3. Build the graph and launch the chat
4. Exercises (14 to 16)
5. Submitting your session log

## 0. Setup: your name, a backend, and its API key
**Objective:** identify yourself for the session log and choose the language model that powers the agent.

Your name and student ID come from `config/.env` — the same git-ignored file as your API key — so nothing identifying is typed into this notebook or saved in its outputs. Fill in `STUDENT_NAME` and `STUDENT_ID` there if you have not already. Both are encrypted into your session log, which is how it gets matched to you however the file is renamed. The backends and the `config/.env` key are the same as in Lab 1 (see `config/.env.example`). **Set up both providers if you can.** Either one answers a tool-calling turn in a few seconds, but a single provider runs out of its per-minute token budget quickly here: every message the agent handles resends the system prompt and all fourteen tool descriptions, so one key alone stalls after a couple of messages. With both configured and fallback on, seven messages in a row go through with no waiting, the chat switches providers when one budget ran out and switches back a minute later once it recovered.

**A second provider helps more than a second key of the same kind.** Groq applies its rate limit to your whole account, so extra Groq keys add no quota. Extra Gemini keys help only when each comes from a separate Google project, since each project has its own limit. One key from each provider is the combination that keeps this notebook moving.

If you have keys for both providers, **enable multi-provider fallback**. When a backend hits a rate limit, or its servers are unreachable or overloaded, the agent retries on another key or provider instead of stopping, and the chat prints a line whenever it switches. The limit you will actually meet is a per-minute one, and it clears within about a minute. The daily quotas are much larger and reset once a day (Groq at midnight UTC, Gemini at midnight Pacific time). It's recommended to start with time to spare, just in case.

You can type `/status` in the chat at any point to see which of your keys are still active, which have hit a rate limit this session, and how large the requests are getting. The chat widget answers that itself: it never reaches the agent, costs no turn, and is not written to your session log. `/keys` does the same thing.

Run the cell below, fill it in, and click **Confirm** before moving on.

In [1]:
from agent_pipeline.backend_widget import select_backend

backend_choice = select_backend()

## 1. Load the saved pipeline
**Objective:** load the agent's tools and the saved pipeline from disk, so this notebook starts from a fresh kernel and still works on exactly what the Master built.

The tools come from two folders. `lab2_tools/inspection/` holds tools that only read what is already saved (the list of representations, the Master's results table, breakdowns of earlier results). They compute nothing new and send no tweets to the model, so they are usable immediately. `lab2_tools/experiments/` holds tools that train a classifier, run retrieval, or show example tweets; those need your explicit approval before they run, through the same request and approval exchange as Lab 1's agentic pipeline, and one approval covers all five of them at once rather than one tool at a time.

Every tool reads `lab2_workspace/`, never this kernel's memory. That is why the agent's numbers can be checked against the Master: a run with the Master's settings reproduces the Master's score to within a prediction or two. If the cell below reports that the workspace is missing, run `DM2026-Lab2-Master.ipynb` through Section 9 first.

In [2]:
import os

from agent_pipeline.session_state import SessionState
from agent_pipeline.workspace_loader import load_workspace_tools
from agent_pipeline.tool_access import build_gated_tools

if not os.path.isfile("lab2_workspace/manifest.json"):
    print("lab2_workspace/ not found. Run DM2026-Lab2-Master.ipynb through Section 9 first, then re-run this cell.")

session = SessionState()
experiment_tools, experiment_problems = load_workspace_tools("lab2_tools/experiments", session)
inspection_tools, inspection_problems = load_workspace_tools("lab2_tools/inspection", session)
tools = build_gated_tools(experiment_tools, session, premade_tools=inspection_tools)
problems = experiment_problems + inspection_problems

print(f"Experiment tools ({len(experiment_tools)}, need your approval to run): {[t.name for t in experiment_tools]}")
print(f"Inspection tools ({len(inspection_tools)}, usable right away): {[t.name for t in inspection_tools]}")
if problems:
    print("\nNot loaded:")
    for fname, message in problems:
        print(f"  {fname}: {message}")

Experiment tools (5, need your approval to run): ['run_classifier_tool', 'show_misclassified_tool', 'retrieve_tool', 'compare_retrieval_tool', 'evaluate_retrieval_tool']
Inspection tools (4, usable right away): ['list_representations_tool', 'show_master_results_tool', 'inspect_result_tool', 'compare_results_tool']


## 2. What the agent can do, and what to check
**Objective:** know the tools well enough to direct them precisely, and know what to verify in the agent's replies before trusting them.

| Tool | Needs approval | What it does |
|---|---|---|
| `list_representations_tool` | no | Lists the five saved representations and the dataset's schema (column names and types, row and label counts). |
| `show_master_results_tool` | no | The Master's classification results table, best first: the baseline to compare against. |
| `inspect_result_tool` | no | Per-emotion precision, recall, and F1, the confusion matrix, and the most frequent confusion for one classifier run. |
| `compare_results_tool` | no | Two or more classifier runs side by side, with the gap between best and worst. |
| `run_classifier_tool` | yes | Trains KNN, a linear SVM, Naive Bayes, or an MLP on one representation and evaluates it on the test tweets. |
| `show_misclassified_tool` | yes | Up to 10 test tweets one run got wrong, with true and predicted emotion and the annotated intensity. |
| `retrieve_tool` | yes | The top *k* training tweets for a query under one retriever (TF-IDF, Word2Vec, or MiniLM). |
| `compare_retrieval_tool` | yes | One query through all three retrievers, with how many documents each pair shares. |
| `evaluate_retrieval_tool` | yes | Label precision@*k* for one retriever over all test queries, with the random baseline. |

The agent also has the five access tools you know from Lab 1 (list, describe, request, grant, discard).

**The first experiment tool you ask for takes three messages of your own, and it does not run until the third.** The graph allows the agent one tool call per message, and granting access is itself a tool call, so approving and running cannot happen together. The approval is a batch: it covers all five experiment tools at once, so this exchange happens once per session, not once per tool:

| Your message | What the agent does | What you should see |
|---|---|---|
| 1. Ask for the step, with the details specified | Calls `request_tool_access_tool` and explains why | A pending request. No result yet. |
| 2. Approve it (for example "yes, go ahead") | Calls `grant_tool_access_tool` | A confirmation that all five experiment tools are now usable. **Still no result:** nothing has run yet. |
| 3. Ask for the run again | Calls the tool itself | The result, with a `result_id` |

After that, every experiment tool runs directly for the rest of the session, one call per message, including the ones you have not asked for yet. The agent will not request and grant in one step, and it will not chain two steps into one message, however the request is phrased.

**If approving seems to do nothing.** Approving produces a confirmation, not a result, so message 2 always looks quiet. When the agent answers with nothing but the tool call it wants next, the chat prints that step instead of a blank reply, as `(Done with that step. Next it wants to run ... Say "go ahead" to run it, or ask for something else.)`, which is your cue to send the next message. If the agent replies with a comment but never calls `grant_tool_access_tool`, repeat the approval. If it tries to run the tool in the same message as the grant, the run is refused (it was not granted yet) and the grant is dropped, so nothing changes at all: approve on its own, and ask for the run in a separate message.

**What to check in every reply.** The agent's prose is a claim about tool output, not the output itself. The tables in the chat come directly from the tool; the sentences around them come from the model. Each time, check:
- that every number it states appears in the tool output above it (the chat marks any number it cannot trace, and the mark is also recorded in your log);
- that it describes the tweets actually shown, not tweets it imagines (compare its summary with the table);
- that a claimed improvement is larger than the differences noise alone would produce; on 347 test tweets, a gap in the third decimal is not a finding;
- that a run with the Master's settings reports a `master_reference_macro_f1` within a prediction or two of its own `macro_f1`, and that you find out why if the gap is wider.

**What your log records.** Each message you send is logged with the tool call it triggered (name and arguments), the tool's result, and the agent's reply, encrypted as it is written. Grading reads that sequence: specific, one-step requests that build on earlier results score well; vague requests, requests copied from this notebook, and approvals given without reading what was proposed do not.

**If the agent's reply is only "(No response came back for that...)".** The tool call already ran and its result is shown above; only the model's comment on it came back empty. Ask it to summarize the result, or continue with your next request. Nothing needs restarting.

**If replies seem to be missing entirely, check your widget versions.** With `ipywidgets` 8.x the chat's input row renders inside the scrollable conversation box, so new replies can appear underneath it and look absent. From the repository root, run `uv sync --locked --python 3.11` to restore the pinned `ipywidgets==7.8.1` environment, then restart Jupyter with `uv run --locked jupyter lab`.

**If the chat does not appear at all, restart JupyterLab itself, not just the kernel.** Now and then the launch cell renders an error or nothing where the chat should be, and restarting the kernel does not clear it, because the stale widget state lives in the browser tab and the running server rather than in the kernel. Close the JupyterLab tab, stop the server in the terminal you started it from (`Ctrl+C`), start it again with `uv run --locked jupyter lab`, and run the notebook from the top. The chat renders normally after that, and nothing you have already logged is lost.

**If the chat reports a rate limit or an unreachable provider, or tells you to wait and send another message.** Every configured key and provider failed in a row, and the chat handed control back to you rather than freezing the notebook. Do not restart or stop the kernel: the retry progress lives only in its memory, and a restart starts the count over. Wait a few minutes and send your next message; the retries continue where they stopped. Type `/status` to see which keys are still active before you spend a turn on a real request.


## 3. Build the graph and launch the chat
**Objective:** connect the tools, the rules, and your chosen model into the agent, and open the chat.

The graph is the one Lab 1's agentic pipeline uses (`agent_pipeline/graph_core.py`): one tool call per message, then a forced plain-text comment on the result, then a stop to wait for you. The system prompt at the top of the cell states the rules the agent works under; read it once, since it defines what you can reasonably ask for. Re-running this cell starts a fresh conversation, and the log keeps the earlier one.

In [3]:
SYSTEM_PROMPT = """You run experiments over the student's saved Lab 2 pipeline. \
DM2026-Lab2-Master.ipynb saved it to lab2_workspace/: five representations \
(tfidf, word2vec, doc2vec, fasttext, minilm) of the same SemEval-2017 emotion \
tweets (anger, fear, joy, sadness), the fitted models, and the Master's own \
results table. Your tools read those files. You never see the dataset itself, \
only small summaries and, when the student asks, at most ten example tweets.

Inspection tools (list_representations_tool, show_master_results_tool, \
inspect_result_tool, compare_results_tool) only read saved results and are \
usable immediately. Experiment tools (run_classifier_tool, \
show_misclassified_tool, retrieve_tool, compare_retrieval_tool, \
evaluate_retrieval_tool) compute new results or send tweets to you, and need \
the student's approval first, granted once as a batch covering all five. \
Calling one before that returns a refusal, not a result. Where the access \
tools say "the student's own tools", they mean these five.

Rules you must follow:
1. Call AT MOST ONE tool per response, and only when the student has asked for \
a specific action with the details it needs.
2. If a request is ambiguous (which representation, classifier, setting or \
query), ask instead of guessing.
3. While the batch is ungranted, call request_tool_access_tool with a reason \
tied to what the student asked, then stop and wait. It takes only that reason \
and proposes all five at once. When the student's next message approves, call \
grant_tool_access_tool as that turn's ONLY tool call: never in the same turn \
as the request, and never in the same turn as the tool being granted, since \
only your first tool call in a turn runs. Never answer an approval with plain \
text alone; make the grant call. That one grant makes all five usable, so say \
so and name the exact request to send next. Once granted, never call \
request_tool_access_tool again. If the student declines, call \
discard_tool_request_tool.
4. Answer what tools exist or what one does with list_available_tools_tool or \
describe_tool_tool, never from a name alone.
5. After a tool returns, report what it shows, quoting the actual values and \
doc_ids rather than paraphrasing them.
6. Every number you state must come from a tool's returned data. Do not \
compute, estimate or invent one, and do not call a difference an improvement \
without stating its size.
7. Refuse to chain several steps in one message; the student directs each step.
8. Refer to an earlier result by its exact result_id.
9. Right after a tool result you get exactly one turn to comment in plain \
text, with no tool call of any kind. That applies only to that turn; the \
student's next message allows a tool call again.
"""

from agent_pipeline.graph_core import build_agent_graph
from agent_pipeline.chat_widget import launch_chat

if "backend" not in backend_choice:
    raise RuntimeError("Pick a backend in the setup cell above and click Confirm before running this one.")
LLM_BACKEND = backend_choice["backend"]
MULTI_PROVIDER_MODE = backend_choice["multi_provider"]
STUDENT_NAME = backend_choice["student_name"]
STUDENT_ID = backend_choice["student_id"]

if MULTI_PROVIDER_MODE:
    from agent_pipeline.llm_fallback import build_fallback_llm
    llm, active_chain = build_fallback_llm(LLM_BACKEND, temperature=0)
    print(f"Fallback chain (in order): {active_chain}")
else:
    from agent_pipeline.llm_backends import build_llm_for_backend
    llm = build_llm_for_backend(LLM_BACKEND, temperature=0)

app = build_agent_graph(tools, SYSTEM_PROMPT, llm, session)

graph_state, logger = launch_chat(
    app, session, log_path="session_logs/lab2_agent_pipeline_session.jsonl.enc",
    student_name=STUDENT_NAME, student_id=STUDENT_ID, track="lab2-agent-pipeline",
    placeholder="Ask the agent to run one experiment... (or type /status to see key rotation status)",
    llm=llm,
)

## 4. Exercises
**Objective:** practice directing controlled experiments through the agent and auditing its reports against tool output.

Answer each exercise in the cell below it, citing the `result_id`s from your conversation. The graded evidence is what you write here together with the logged conversation that produced it.

### **>>> Exercise 14 (take home):**
**Purpose:** beat a number the Master already obtained, and claim no more than the margin allows.

Pick any row of the Master's results table and try to beat its macro F1. Reproduce it first (same representation, classifier and settings), and check your `macro_f1` against the `master_reference_macro_f1` beside it. That is your baseline. Then make your attempts, one run per message. Ask the agent what `run_classifier_tool` can vary, and count the table: seventeen rows, but five representations times four classifiers is twenty. Four or five runs is enough; finish with `compare_results_tool`.

In the cell below, give your baseline and best attempt with their `result_id`s and say whether you beat the Master. The Master read 0.006 macro F1 as three of the 347 test predictions (Section 7); convert your margin the same way and say whether it is a finding or a coin flip. **If nothing beat the Master's setting, say so and show the runs**. Close with two sentences: one quoting an agent claim you checked yourself, and one on whether your best score is a fair estimate for new tweets, given that you chose it on the test set.

*What to check in the agent's output:* each run changed only what you asked, and its summary of the `compare_results_tool` table matches the table.

In [ ]:
#Answer here 
"""
**Baseline**

I reproduced the Master's baseline for TF-IDF + KNN ($k=5$) in `classify_3`. It got a macro F1 of 0.6536 (with 0.6571 accuracy), which matches the Master's reference score.

**My Attempts**

I changed one thing at a time and compared all runs in `compare_8`. Here is how my runs compared against the baseline of 0.6536 across the 347 test tweets (the number of predictions is estimated as the macro F1 gap × 347):

* `classify_4` (TF-IDF, $k=15$): Macro F1 of 0.6486, down by 0.0050 (about 2 predictions worse).
* `classify_5` (TF-IDF, $k=3$): Macro F1 of 0.6247, down by 0.0289 (about 10 predictions worse).
* `classify_7` (TF-IDF, $k=8$): Macro F1 of 0.6182, down by 0.0354 (about 12 predictions worse).
* `classify_6` (MiniLM, $k=5$): Macro F1 of 0.6054, down by 0.0482 (about 17 predictions worse).

**Did I beat the Master?**

No. None of my runs scored higher than the baseline (`classify_3` at 0.6536). The closest one, `classify_4` ($k=15$), was 0.0050 lower. Out of 347 test tweets, that's only a difference of about 2 predictions, basically a toss-up, similar to the 3-prediction tie mentioned in Section 7 of the Master notebook.

However, the other three runs dropped by 10 to 17 predictions, which is unlikely to be random noise alone. Changing only $k$ (to 3, 8, or 15) or switching the representation to MiniLM didn't improve the results. On the bright side, `classify_6` matched the Master's MiniLM + KNN score (0.6054) exactly, which confirms my pipeline is set up properly.

**Fact-checking the agent's claim**

For `classify_3`, the agent reported "Master Reference Macro F1: 0.6536", the same as my run's Macro F1. I double-checked this against the Master's results table for TF-IDF + KNN ($k=5$), and the number is spot on.

**Is my best score a fair estimate for new tweets?**

Not really. Because I tested several settings on the exact same 347 test tweets and simply picked the winner, this score is likely a bit too optimistic. My best score is also just the Master's own setting, which was itself chosen after looking at test results. Choosing parameters after seeing the test results introduces bias. To get a truly fair estimate, I would need to evaluate the model on unseen data that wasn't used to make that choice, like a separate validation set or through cross-validation. 
"""

### **>>> Exercise 15 (take home):**
**Purpose:** judge what three representations retrieve for the same query, and weigh one query's verdict against an average over 347.

1. Write two queries of your own about a topic this dataset contains. Predict which of the three retrievers will serve each better, and report where you were wrong.
2. Have the agent run `compare_retrieval_tool` on each, one per message. Read the three tables of retrieved tweets yourself before reading the agent's summary.
3. Have it run `evaluate_retrieval_tool` for TF-IDF and MiniLM at the same *k*. It repeats the Master's Section 8.2 computation, so check its numbers against the table you printed there.

In the cell below, for each query: which retriever's list would you use, citing `doc_id`s, and does the agent's summary of it match the table? What does Word2Vec's list say about that representation? Your queries may make one retriever look far better than another while precision@*k* separates them by a few hundredths: explain how both can be true.

*What to check in the agent's output:* the overlap counts it quotes match the `overlap` table; it does not present one query's result as a general conclusion.

In [5]:
#Answer here 
""" 
**A quick note on my predictions**

I forgot to write down my formal predictions before running the tools, so the "expectations" below are just what I originally had in mind when picking each query, written down afterward. They aren't pre-logged predictions, though I kept them completely honest and didn't tweak them to fit the results.

## Query 1: "I'm nervous about what is going to happen in the future." (`compare_retrieval_12`, $k=5$)

**Expectation**

I expected MiniLM to handle this best since it captures the overall meaning of a sentence. The query expresses one clear, consistent feeling without any plot twists, so tweets with a similar vibe should land right next to it.

**Result**

I was right. I'd definitely go with MiniLM's list here. All five tweets (20039, 20458, 20195, 20001, 20066) are labeled fear and focus on worry or the future, with 20066 ("impending doom... #anxiety") being a particularly great match. One minor catch is that a fear label doesn't always guarantee relevance; tweet 20195 actually reads pretty positively and only got pulled in because of its #nervous hashtag.

Meanwhile, TF-IDF only caught two relevant tweets (20039 and 20458) and filled the rest with noise based on common words like "going," "what," and "happen" (20890, 40025, 40213). Word2Vec's list was almost entirely off-topic (only 20196 mentions being fearful). Looking at the overlap, TF-IDF and MiniLM shared 2 documents (20039, 20458), while every other pair had zero overlap, which matches my manual count from the results.

**What surprised me**

In my first exploratory runs (`retrieve_9` to `retrieve_11`), I accidentally left angle brackets `< >` around the query, assuming they wouldn't make a difference. Once I cleaned them out in `compare_retrieval_12`, MiniLM swapped out two of its five tweets (30341 and 20141 were replaced by 20001 and 20066) and shifted all its similarity scores, whereas TF-IDF and Word2Vec returned the exact same lists. It appears that MiniLM pays attention to punctuation, while the other two ignore it. (All my conclusions here are based on the clean run.)

## Query 2: "my phone service is down and customer support is not helping" (`compare_retrieval_13`, $k=5$)

**Expectation**

I figured TF-IDF would win this one because the query uses specific keywords that real tweets often share, and TF-IDF is great at rewarding exact word matches.

**Result**

I was only partly right. TF-IDF did find tweet 40025 ("my service is down"), which was the closest match to the actual situation and something MiniLM missed. However, TF-IDF's top result (30106) was actually praising a store's customer service (joy), and 40152 was about emotional support rather than a phone carrier, so it matched the words while completely missing the tone.

MiniLM's list stayed on topic for all five tweets (20636, 10158, 20470, 10015, 40022), though the emotion labels were mixed (2 fear, 2 anger, 1 sadness) and it included two near-duplicate pairs (20636/20470 and 10158/10015). Overall, I'd still pick MiniLM's list, even though TF-IDF beat it on that single best match (40025). Again, TF-IDF and MiniLM overlapped on 2 tweets (20470, 20636) with zero overlap elsewhere, and Word2Vec returned completely random topics like football, poetry, and police news (e.g., 20822, 40692, 10191).

**What surprised me**

I didn't realize how often "customer service" shows up in positive tweets, which is why matching exact keywords wasn't enough for TF-IDF to take the lead. I was also surprised by Word2Vec's very high similarity scores (0.9022 to 0.9476). They operate on a totally different scale than MiniLM's (0.5177 to 0.5565), meaning you can't directly compare raw similarity scores across different retrievers.

## Did the agent's output match the tables?

For both comparisons, the agent only gave raw tables, overlap numbers, and emotion counts without any summary or interpretation. I double-checked those overlap and emotion counts against the tables myself, and everything matched up.

## What does Word2Vec's list say about that representation?

Across both queries, Word2Vec had zero overlap with the other two retrievers and pulled mostly unrelated tweets. This lines up with its precision@5 score in the Master notebook (0.284), which is barely better than random guessing (0.256). Averaging word vectors across an entire tweet seems to blend everything together until most tweets look the same, washing out specific keywords like "nervous" or "service." This is based on only two queries plus the Master's average, so it isn't definitive proof on its own, but it is consistent with the overall benchmark.

## Overall retrieval evaluation (`evaluate_retrieval_tool` at $k=5$)

Across all 347 queries (1,735 retrieved tweets total), the precision@5 results match the Master's Section 8.2 table (0.523, 0.549, and 0.256):

* MiniLM (`evaluate_retrieval_15`): 0.5487
* TF-IDF (`evaluate_retrieval_14`): 0.5233
* Random baseline (both): 0.2563

The gap between MiniLM and TF-IDF is just 0.0254, which works out to roughly 44 extra hits out of 1,735 retrieved tweets.

## How can individual queries look so different when precision@5 only differs by 0.025?

1. **Single cases vs. overall averages:** The precision@5 table averages performance across 347 queries, whereas I only looked closely at two. One retriever can easily dominate a specific query and lose on another (like TF-IDF catching 40025 when MiniLM missed it), and the average smooths those swings out. Broken down by emotion, MiniLM leads on fear (0.5836 vs. 0.5055) and joy (0.5342 vs. 0.5013), while TF-IDF edges ahead on anger (0.5667 vs. 0.5571) and sadness (0.5243 vs. 0.5027).
2. **Precision@k checks emotion labels, not topics:** A retrieved tweet counts as a "hit" purely based on whether its emotion label matches the query's, not whether the topic makes sense. In Query 2, all five of MiniLM's tweets were genuinely about service issues, yet they carried three different emotion labels, meaning its topical relevance wasn't fully rewarded. Conversely, a completely off-topic tweet like 30106 would still score a hit if its emotion label happened to match by coincidence.
3. **Both are far ahead of random chance:** Since both retrievers score more than double the 0.2563 random baseline, the 0.025 gap between them is small relative to how far both are from random guessing. 
"""

SyntaxError: incomplete input (1078743744.py, line 2)

### **>>> Exercise 16 (take home):**
**Purpose:** treat an agent's explanation of model errors as a set of claims to test against evidence.

In the Master's Exercise 8 you explained your best model's most frequent confusion from the confusion matrix alone, yet never saw a misclassified tweet. 

One request per message: have the agent run a classifier (Exercise 14's best, or the Master's TF-IDF linear SVM) so there is a `result_id`, break it down with `inspect_result_tool`, then show up to five tweets from one confusion with `show_misclassified_tool` and explain why the model got them wrong. Pick the confusion from the whole matrix yourself: `most_confused` names one direction, where Exercise 8 had you add up both.

In the cell below, say which confusion you chose and why, and put the agent's explanation next to your Exercise 8 paragraph. Which of its sentences say something your tweet-blind paragraph could not have said, and which tweets support them (cite `doc_id`s). For the rest, say what in the table could have shown them wrong.

*What to check in the agent's output:* it describes the tweets in the table and not tweets it invented, and refers to them by `doc_id`.

In [ ]:
#Answer here 
""" 
## Which confusion I chose, and why

I went with the **anger / fear** pair. When adding both directions together, it is the largest source of error in the confusion matrix: 7 anger tweets predicted as fear and 8 fear tweets predicted as anger, making up 15 of the 48 total errors (for comparison, anger/sadness and fear/sadness have 10 each). This is also the pair I analyzed in Exercise 8, which lets me compare the agent's explanations directly against my own findings. I chose it from the whole matrix rather than from `most_confused`, which flags only a single direction (fear → anger, count 8), though both approaches lead to the same pair.

## What I ran

- `classify_1`: a linear SVM on TF-IDF features (C=1.0), with a macro F1 of 0.8621, which matches `master_reference_macro_f1` (0.8621).
- `inspect_result_tool` on `classify_1` produced the same confusion matrix as my Exercise 8 in every cell.
- `misclassified_3` returned 5 of the 8 fear → anger errors, and `misclassified_4` returned 5 of the 7 anger → fear errors. It is unclear how the tool selects which examples to display, so I treated these as a subset rather than a representative sample.

Matching the tool's `doc_id`s to my Exercise 8 tweets by text confirms they are the same tweets: 21151 = 88, 21165 = 102, 21189 = 126, 21221 = 158, 21233 = 170; and 10861 = 4, 10867 = 10, 10879 = 22, 10905 = 48, 10907 = 50.

**A note on my Exercise 8.** The exercise assumes the Exercise 8 paragraph was written without seeing any tweets. Mine was not: I had already printed all 15 misclassified tweets, checked their intensity, and computed per-word weights with `explain()` (for the first three anger → fear tweets only). So instead of asking what the agent said that I could not have said, I ask what it noticed that I missed, and what my analysis caught that the agent could not.

## My Exercise 8 paragraph (for comparison)

> The model most often confuses anger and fear: 15 of its 48 errors (7 anger tweets predicted as fear, 8 fear tweets predicted as anger), so the confusion is close to symmetric. Anger/sadness and fear/sadness follow with 10 errors each, so the lead is small. Reading these 15 tweets suggests several causes. TF-IDF only sees which words occur, so the classifier relies on weights learned from the training labels, and some of them are weak or misleading: in tweet 22 the strongest pull toward fear is "was", a neutral word that appears in 6.9% of training fear tweets vs 4.7% of anger tweets, and "trash", which sounds angry, occurs in only 5 training tweets, 4 of them fear, yet pulls tweet 10 toward fear. Some clues never reach the model: the emoji in tweet 50 and "atrocious" in tweet 10 have no TF-IDF column, and the sarcasm or exaggeration in tweets 4 and 170 probably depends on word order, which a bag of words discards. Some errors are also not really the model's fault: tweets 158 and 159 are the same message (one has an extra #awful), so one message is counted twice, and several fear-labelled tweets (88, 158, 159) read to me as anger, so the labels themselves are ambiguous. Misclassified tweets showed no clear difference in intensity (mean 0.45 vs 0.49 for all test tweets), and since all of this comes from 15 examples, it is a plausible explanation, not proof.

## Where the agent's explanation adds something

- **10905** ("...accidentally scared the #cat whilst #screeching"): the agent points out the words "scared" and "screeching", which explains why the model predicted fear for an anger-labelled tweet. I had missed this. It makes sense, though I have not run `explain()` on this tweet to confirm those words carry the weight.
- **10907** ("...sorted into HUFFLEPUFF 😡😡😡"): the agent notes that the anger comes entirely from the emoji while the words are neutral. My own check goes further: `"😡" in feature_names` is False, so the emoji is not just down-weighted, it has no TF-IDF column at all.
- **10879** ("Where was my invite?"): the agent reads this as a passive-aggressive question with no explicit angry word. This fits my Exercise 8 weights: the strongest pull toward fear is "was" (+0.34), while "oh" (−0.21) and "where" (−0.11) pull toward anger.
- **21221** (the Lufthansa complaint about money): the agent reads this as a standard customer-service complaint, which points to anger. It does not question the fear label, but its reading supports my own suspicion that the label may be wrong.

## What my analysis caught that the agent missed

The agent treats every label as correct. In my Exercise 8, I noted that 21151 (88), 21221 (158) and its near-duplicate 159 read as anger, so some of these "errors" may be label problems. Tweets 158 and 159 are the same message (only `#awful` differs), which counts one tweet twice. Because the agent never sees the model's feature weights (`coef_`), and did not use the `intensity` scores that the tool showed it, it misses these structural issues.

## Claims I could test, and what the data shows

- **"atrocious" and "#trash" pull 10867 toward fear: partly wrong.** `"atrocious" in feature_names` is False, so the model cannot use that word. "#trash" does pull toward fear (+0.25; 4 of its 5 training occurrences are fear), but the strongest pull is "please" (+0.26), which the agent did not mention.
- **"Words like prison, lose your job and ruin often co-occur with fear in general corpora" (10861): partly right words, wrong reasoning.** The model never saw a general corpus, only 3,606 training tweets. The prediction does lean toward fear, but because "job" appears in 9 training fear tweets against 2 anger tweets (+0.16), and "ruin" (+0.10) also pulls toward fear. "prison" and "lose" are not among the top pulls.
- **Fear is "masked" in 21151 and 21165 ("underlying apprehension", "mild anxiety"): unsupported.** Nothing in the table backs this up. I read 21151 as anger, so the label may be the problem. 21165 has a low intensity (0.25), but low intensity does not show hidden fear.
- **Sarcasm in 21233 pushes it toward anger: not a mechanism a bag of words can have.** TF-IDF cannot see sarcasm, which depends on word order. The model could only be reacting to individual words, but I did not run `explain()` on this tweet, so I cannot say which.
- **"Unfollowed without hesitation" signals spite (21189): a guess.** Nothing in the feature weights supports it, and the agent also skipped the `&lt;3` (an escaped heart emoticon) at the end of the text.

## A short observation on intensity

On the tweets shown, the fear → anger errors have a mean intensity of about 0.34 (range 0.167 to 0.542) and the anger → fear errors about 0.57 (range 0.438 to 0.708). Across all 15 errors in Exercise 8 the mean was 0.45 against 0.49 for all test tweets. Five tweets per direction, chosen in an unknown way, are too few to conclude anything. It fits the idea that some fear labels are weak, but it is only a lead.

## Caveat

All of this covers 10 of the 15 errors, and the agent's explanations are plausible readings of the text, not proof of how the model actually works. 
"""

## 5. Submitting your session log
**Objective:** confirm the log exists before you submit it.

Your conversation is logged turn by turn to `session_logs/lab2_agent_pipeline_session.jsonl.enc`, encrypted with the course's grading public key as it is written. You cannot read or edit it; upload it exactly as produced, together with this notebook (Exercises 14 to 16 answered) and the Master notebook's log, `session_logs/lab2_main_session.jsonl.enc`. `session_logs/` is git-ignored, so neither log will be in your repository: both are handed in on NTU COOL, inside the ZIP.

For the complete lab submission, push your notebooks to your repository and submit its link plus the ZIP on NTU COOL by **November 25 at 11:59 pm**. If your ZIP turns out to be missing a log, you can upload it until **November 26 at 11:59 pm**. See `README.md` for the ZIP contents.

In [4]:
import os

log_path = "session_logs/lab2_agent_pipeline_session.jsonl.enc"
if os.path.exists(log_path):
    print(f"Log file present: {log_path} ({os.path.getsize(log_path)} bytes)")
else:
    print("No log yet. Send at least one message in the chat above first.")

Log file present: session_logs/lab2_agent_pipeline_session.jsonl.enc (93492 bytes)
